# Parallel & Concurrent Programming

## Calling a Function with an Argument in Parallel

Call the function `f` that gives information about the process executing it in parallel using a `multiprocessing.Pool`.

In [ ]:
import os
import multiprocessing


def f(n: int) -> None:
  print(f"Process {n}")
  print(f"Parent process ID: {os.getppid()}")
  print(f"Process ID: {os.getpid()}")


# Your code here

### Solution

In [ ]:
import os
import multiprocessing


def f(n: int) -> None:
  print(f"Process {n}")
  print(f"Parent process ID: {os.getppid()}")
  print(f"Process ID: {os.getpid()}")


with multiprocessing.Pool() as pool:
  pool.map(f, range(10))

## Calling a Function in Parallel with Constant Arguments

Use [`functools.partial`](https://docs.python.org/3/library/functools.html#functools.partial) and adapt the previous code to call `f` with `n` ranging from 0 to 9 and `verbose` always set to `False`.

In [ ]:
import os
import multiprocessing
import functools


def f(n: int, verbose: bool) -> None:
  if verbose:
    print(f"Process {n}")
    print(f"Parent process ID: {os.getppid()}")
    print(f"Process ID: {os.getpid()}")
  return n * 2


# Your code here

### Solution

In [ ]:
import os
import multiprocessing
import functools


def f(n: int, verbose: bool) -> None:
  if verbose:
    print(f"Process {n}")
    print(f"Parent process ID: {os.getppid()}")
    print(f"Process ID: {os.getpid()}")
  return n * 2


with multiprocessing.Pool() as pool:
  results = pool.map(functools.partial(f, verbose=False), range(10))

print(results)

## Downloading Multiple Files Simultaneously, with a Single Argument

Use a `multiprocessing.pool.ThreadPool` to download multiple files simultaneously. Initially, we will download the Wikipedia random page 10 times: `https://en.wikipedia.org/wiki/Special:Random`. The parallelized function will simply retrieve an integer and store the download result in an `articles` folder under `{n}.html` if `n` is the integer.

For file downloading, you can use the following code:

```python
with urllib.request.urlopen(url) as response, path.open("wb") as fh:
    shutil.copyfileobj(response, fh)
```

where `url` is the URL to download and `path` is the path to write the file.

In [ ]:
import multiprocessing.pool
import pathlib
import shutil
import urllib.request


articles = pathlib.Path("articles")
articles.mkdir(exist_ok=True)
url = "https://en.wikipedia.org/wiki/Special:Random"


# Your code here

### Solution

In [ ]:
import multiprocessing.pool
import pathlib
import shutil
import urllib.request


articles = pathlib.Path("articles")
articles.mkdir(exist_ok=True)
url = "https://en.wikipedia.org/wiki/Special:Random"


def download_article(n: int):

  with urllib.request.urlopen(url) as response, \
          (articles / f"{n}.html").open("wb") as fh:
      shutil.copyfileobj(response, fh)


with multiprocessing.pool.ThreadPool() as pool:
  results = pool.map(download_article, range(10))

## Downloading Multiple Files Simultaneously, with Two Arguments

Similar to the previous exercise, we want to download multiple pages at the same time. This time we want to give our worker a URL and a path to write to, rather than just an integer.

Adapt the previous code to download `to_download`.

In [ ]:
import pathlib
import multiprocessing.pool
import shutil
import urllib.request


downloads = pathlib.Path("downloads")
downloads.mkdir(exist_ok=True)

to_download = (
    ("https://docs.python.org/fr/3/", downloads / "python-docs.html"),
    ("http://pythontutor.com/", downloads / "python-tutor.html"),
    ("https://www.google.com/", downloads / "google.html"),
)


# Your code here

### Solution

In [ ]:
import pathlib
import multiprocessing.pool
import shutil
import urllib.request


downloads = pathlib.Path("downloads")
downloads.mkdir(exist_ok=True)

to_download = (
    ("https://docs.python.org/fr/3/", downloads / "python-docs.html"),
    ("http://pythontutor.com/", downloads / "python-tutor.html"),
    ("https://www.google.com/", downloads / "google.html"),
)


def download(url: str, path: pathlib.Path) -> None:

  with urllib.request.urlopen(url) as response, path.open("wb") as fh:
      shutil.copyfileobj(response, fh)


with multiprocessing.pool.ThreadPool() as pool:
  pool.starmap(download, to_download)

## Creating a Producer/Consumer Architecture

In [ ]:
import multiprocessing
import time


def consumer(queue: multiprocessing.Queue) -> None:
  while True:
    item = queue.get()
    if item is None:
      break
    print(item)


def producer(queue: multiprocessing.Queue) -> None:
  for i in range(10):
    queue.put(i)
  queue.put(None)


if __name__ == "__main__":
  queue = multiprocessing.Queue()
  consumer = multiprocessing.Process(target=consumer, args=(queue,))
  producer = multiprocessing.Process(target=producer, args=(queue,))
  consumer.start()
  producer.start()
  consumer.join()
  producer.join()

## Performing Two Different Tasks in Parallel with a `Pool`

In [ ]:
import multiprocessing


def a(i: int) -> int:
  return i * 2


def b(i: int) -> int:
  return i ** 2


if __name__ == "__main__":
  with multiprocessing.Pool() as pool:
    a_results = pool.map_async(a, range(10))
    b_results = pool.map_async(b, range(10))
    print(a_results.get())
    print(b_results.get())